# ref-ball — Landing Foul VideoMAE Fine-tuning

End-to-end fine-tuning of `videomae-base-finetuned-kinetics` on the 284-clip landing-foul ground truth, targeting **≥85% precision on YES** (the gate for Step 11–12).

**Runtime: GPU required.** `Runtime → Change runtime type → T4 GPU` (or A100 if available). CPU will not finish.

**Everything needed is in the repo.** `git clone` supplies the training script, config, the ground-truth CSV, the clip manifest (NBA CDN URLs), and the fixed train/val split. Clips (8.4 GB) are **not** in git — they're re-downloaded from the NBA CDN in §3. No Google Drive staging required.

## 0. Verify GPU

In [ ]:
import torch, subprocess
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Runtime → Change runtime type → T4 GPU, then rerun.')
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout)

## 1. Install dependencies

Colab ships torch (CUDA) + numpy + pandas. We pin `transformers`, `opencv-python-headless`, `scikit-learn`.

In [ ]:
# Install only what's missing. Do NOT upgrade pandas/pyarrow/sklearn — Colab pins them
# (google-colab requires pandas==2.2.2); a blanket -U breaks the runtime.
!pip install -q -U transformers opencv-python-headless tqdm
import transformers, cv2, sklearn, pandas
print('transformers', transformers.__version__, '| sklearn', sklearn.__version__, '| cv2', cv2.__version__, '| pandas', pandas.__version__)
assert pandas.__version__.startswith('2.2'), f'pandas {pandas.__version__} will break google-colab; pin pandas==2.2.2'

## 2. Clone the repo

Gets committed code (`config.py`, `config/`, `src/landing_foul_video_dataset.py`, etc.) and the tracked ground-truth CSVs.

In [ ]:
import os, pathlib
if not pathlib.Path('ref-ball').exists():
    !git clone --depth 1 https://github.com/hirememorey/ref-ball.git
else:
    print('ref-ball/ already present')
os.chdir('ref-ball')
print('cwd:', os.getcwd())
!echo '--- tracked data present? ---'
!test -f data/landing_foul_ground_truth.csv && echo "ground_truth.csv: OK ($(wc -l < data/landing_foul_ground_truth.csv) lines)" || echo MISSING
!test -f src/landing_foul_video_dataset.py && echo "video_dataset.py: OK" || echo MISSING
!test -f src/landing_foul_video_finetune.py && echo "finetune.py: OK (in repo)" || echo "finetune.py: absent (will stage from Drive in §3)"

## 3. Verify cloned artifacts

Confirm the script, manifest, and split all came down with the clone (they're committed to the repo).

In [ ]:
import pathlib
checks = {
    'src/landing_foul_video_finetune.py': 'training script',
    'src/landing_foul_video_dataset.py':  'dataset/clip utilities',
    'data/landing_foul_ground_truth.csv': 'ground-truth labels',
    'data/processed/landing_foul_manifest.json': 'clip manifest (CDN URLs)',
    'data/processed/landing_foul_split.json': 'fixed train/val split',
}
ok = True
for path, label in checks.items():
    present = pathlib.Path(path).exists()
    ok &= present
    print(f"{'OK ' if present else 'MISSING '} {path}  ({label})")
if not ok:
    raise SystemExit('Missing artifacts — check the clone in §2.')

In [ ]:
# (No Google Drive staging needed — all artifacts are committed to the repo.)

## 4. Download clips from NBA CDN (~10 min)

Re-fetches the 284 YES/NO clips at 960×540 (or higher) from the manifest URLs. Verified live 2026-06-30.

In [ ]:
import os, glob
PYTHONPATH = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = '.:' + PYTHONPATH
!python src/landing_foul_video_dataset.py download
n = len(glob.glob('data/clips/landing_foul/*.mp4'))
print(f'\nDownloaded clips: {n} (expect 284)')
if n < 280:
    raise SystemExit(f'Only {n} clips — re-run this cell to resume (download skips existing).')

## 5. Configure hyperparameters

Defaults match `documents/development/HANDOFF.md` Step 10b. Edit the form fields, then run §6.

In [ ]:
phase           = 'two-phase'   #@param ['head','finetune','two-phase']
head_epochs     = 5             #@param {type:'integer'}
finetune_epochs = 15            #@param {type:'integer'}
head_lr         = 1e-3          #@param {type:'number'}
finetune_lr     = 2e-5          #@param {type:'number'}
unfreeze_layers = 4             #@param {type:'integer'}
batch_size      = 4             #@param {type:'integer'}
temporal_window = '0.0,1.0'     #@param {type:'string'}
jitter          = 6             #@param {type:'integer'}
dropout         = 0.4           #@param {type:'number'}
weight_decay    = 0.01          #@param {type:'number'}
yes_weight      = 1.0           #@param {type:'number'}
patience        = 6             #@param {type:'integer'}
seed            = 42            #@param {type:'integer'}
max_train_batches = 0           #@param {type:'integer'}  # 0 = unlimited; set >0 for a quick smoke
max_val_batches   = 0           #@param {type:'integer'}
print('Configured. Run §6.')

## 5b. Build the frame cache (run once, ~5–10 min)

Decoding 1080p MP4s is the dominant cost (~7s/clip); re-decoding every epoch makes a 20-epoch run take ~10 hours. This cell decodes every clip **once** and stores 32 frames/clip at 256×256 (~1.8 GB on Colab disk). After this, each epoch is ~1–2 min instead of ~28 min.

**If you change `temporal_window` in §5, re-run this cell** — the cache stores frames for the configured window only.

In [ ]:
cache_frames = 32  #@param {type:'integer'}
cache_size   = 256  #@param {type:'integer'}
!python src/landing_foul_video_finetune.py --build-cache \
    --temporal-window "{temporal_window}" \
    --cache-frames {cache_frames} --cache-size {cache_size}
import numpy as np
d = np.load('data/processed/landing_foul_frames.npz')
print('cache shape:', d['frames'].shape, '| ~%.2f GB' % (d['frames'].nbytes / 1e9))

## 6. Run fine-tuning

Streams per-epoch loss + val precision/recall. Checkpoint saved to `data/processed/landing_foul_video_best.pt`; metrics to `landing_foul_video_metrics.json`. Best model selected by **val precision on YES**.

In [ ]:
!python src/landing_foul_video_finetune.py \
    --phase {phase} --head-epochs {head_epochs} --finetune-epochs {finetune_epochs} \
    --head-lr {head_lr} --finetune-lr {finetune_lr} --unfreeze-layers {unfreeze_layers} \
    --batch-size {batch_size} --temporal-window "{temporal_window}" --jitter {jitter} \
    --dropout {dropout} --weight-decay {weight_decay} --yes-weight {yes_weight} \
    --patience {patience} --seed {seed} --device cuda \
    --max-train-batches {max_train_batches} --max-val-batches {max_val_batches}

## 7. Save outputs back to Drive

Copies the best checkpoint + metrics JSON to `My Drive/ref-ball/outputs/` so they survive Colab session teardown.

In [ ]:
import shutil, pathlib, json

# Try Drive; fall back to offering a local download if Drive isn't mounted.
out = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    out = pathlib.Path('/content/drive/MyDrive/ref-ball/outputs')
    out.mkdir(parents=True, exist_ok=True)
except Exception as e:
    print('Drive unavailable, will offer a local download instead:', e)

for f in ['data/processed/landing_foul_video_best.pt',
          'data/processed/landing_foul_video_metrics.json']:
    p = pathlib.Path(f)
    if not p.exists():
        print('missing:', f)
        continue
    if out is not None:
        shutil.copy2(f, out / p.name)
        print('saved to Drive:', out / p.name)

if out is None:
    try:
        from google.colab import files
        files.download('data/processed/landing_foul_video_best.pt')
        files.download('data/processed/landing_foul_video_metrics.json')
    except Exception as e:
        print('Could not trigger download:', e)

m = json.load(open('data/processed/landing_foul_video_metrics.json'))
print('\n=== RESULT ===')
print('best epoch:', m['best']['epoch'], '| phase:', m['best']['phase'])
print('val precision (YES):', round(m['best']['val_precision_yes'], 3), '(gate >= 0.85)')
print('val recall    (YES):', round(m['best']['val_recall_yes'], 3), '(gate >= 0.70)')
print('confusion:', m['best']['confusion'])
print('gate verdict:', m['gate']['verdict'])

## 8. (Optional) Evaluate-only on a saved checkpoint

Reload a checkpoint (e.g. one saved to Drive) and re-run val eval + threshold sweep without retraining.

In [ ]:
checkpoint = 'data/processed/landing_foul_video_best.pt'  #@param {type:'string'}
!python src/landing_foul_video_finetune.py --evaluate-only \
    --checkpoint {checkpoint} --device cuda --batch-size 8 \
    --temporal-window "{temporal_window}"

## Decision tree after the run

| val precision (YES) | Next step |
|---|---|
| ≥ 0.85 AND recall ≥ 0.70 | **Gate cleared** → Step 11 (sample 100–150 clips/official) → Step 12 (ANOVA) |
| 0.75–0.84 | Try narrowing `temporal_window` (e.g. `0.3,0.9`), or increase `unfreeze_layers` to 6–8, then rerun |
| 0.55–0.74 | Hybrid: LLM pre-filter (98% recall) → manual review of predicted-YES |
| < 0.55 | Pose estimation (MediaPipe) or scale manual classification |

Note: with 57 val clips, 85% precision ≈ tolerating ~4 false positives — high variance. If a single split looks promising, re-run with a different `seed` and/or implement 5-fold CV for a stable estimate before declaring the gate cleared.